In [16]:
import xarray as xr
import matplotlib.pyplot as plt
import os
import cartopy
import numpy as np
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.crs as ccrs
import matplotlib as mpl
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import datetime
import cdsapi
import zipfile
import pandas as pd
import requests
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from IPython.display import display

In [17]:
product_parameters = {
    'IMARS': {
        'satellite'           : 'METOPC',   # make sure to use the correct satellite name as per the manifest file
        'instrument'          : 'IASI', # make sure to use the correct instrument name as per the manifest file
        'algorithm'           : 'IMARS',    # trying to make this generic enough. most datasets only have a generic 'ALG'
        'version'             : 'v7.1ALL',  # make sure to use the correct version as per the manifest file
        'delivery_start'      : pd.Timestamp('01/07/2025'), # start date of *this* delivery
        'delivery_end'        : pd.Timestamp('31/12/2025'),
        'collection_start'    : pd.Timestamp('01/06/2020'), # start date of the *whole* dataset
        'collection_end'      : pd.Timestamp('31/12/2025'),
        'temporal_resolution' : 'MS', # frequency_code = 'MS' # 'D' for daily, 'MS' for monthly, 'AS' for annual
        'parameter_list'      : ['D_AOD550'],
        'parameter_code'      : 'DAOD',
    },
}


# product_parameters = {
#     'XCO2_OBS4MIPS': {
#         'satellite'           : 'MERGED',   # make sure to use the correct satellite name as per the manifest file
#         'instrument'          : 'MERGED', # make sure to use the correct instrument name as per the manifest file
#         'algorithm'           : 'OBS4MIPS',    # trying to make this generic enough. most datasets only have a generic 'ALG'
#         'version'             : 'v4.6',  # make sure to use the correct version as per the manifest file
#         'GHG_name'            : 'XCO2',  # XCO2 or XCH4
#         'delivery_start'      : pd.Timestamp('01/01/2003'), # start date of *this* delivery
#         'delivery_end'        : pd.Timestamp('31/12/2023'),
#         'collection_start'    : pd.Timestamp('01/01/2003'), # start date of the *whole* dataset
#         'collection_end'      : pd.Timestamp('31/12/2023'),
#         'temporal_resolution' : 'MS', # frequency_code = 'MS' # 'D' for daily, 'MS' for monthly, 'AS' for annual
#         'parameter_list'      : ['xco2'],
#     },
# }

In [ ]:
# Delivery settings

manifest_file = 'https://wdc.dlr.de/C3S_312b_Lot2/manifest_C3S_312b_Lot2_AER_L3_2026-03-20.txt'

product = 'IMARS' # change this to the product name above to cycle through the various data collections delivered.
satellite           = product_parameters[product]['satellite']
instrument          = product_parameters[product]['instrument']
algorithm           = product_parameters[product]['algorithm']
version             = product_parameters[product]['version']
delivery_start      = product_parameters[product]['delivery_start']
delivery_end        = product_parameters[product]['delivery_end']
collection_start    = product_parameters[product]['collection_start']
collection_end      = product_parameters[product]['collection_end']
temporal_resolution = product_parameters[product]['temporal_resolution']
parameter_list      = product_parameters[product]['parameter_list']
parameter_code      = product_parameters[product]['parameter_code']

# options for download
download_full_dataset = False # download full CDR or not
download_delivery     = True # download files for this delivery
# directory where we will store the datasets
work_dir = f'/Users/cxjo/Documents/Datasets/AER/{product}/'
clean_workdir = False        # set if we should delete all the files in the work dir before we start or not

if not os.path.exists(work_dir):
    print(work_dir)
    os.makedirs(work_dir)

if (clean_workdir):
    os.remove(f'{work_dir}/*') # clean directory before we start working (if desired)


# Download the manifest file for this product and save it in the working directory
local_manif_fname = os.path.basename(manifest_file)
r = requests.get(manifest_file)
with open(local_manif_fname, 'w') as file:
    file.write(r.text)

def check_files_in_manifest(date_start, date_end,temporal_resolution,satellite, instrument, algorithm,parameter_code, version, manifest_file):
    """
    Inputs:   Date_Start, date_end, temporal_resolution (all directly from the delivery note); satellite, instrument, algorithm, version (deduced from the delivery note and product documentation);
              manifest_fle (from delivery note)
    Outputs:  existing_files a list of file dates, in string format of actual files delivered and verified.
    Function: checks the list of files reported in the manifest, against the actual datasets in the folder, producing a list of verified and missing files, 
              and producing an object containing a list of verified files.
    """
    # Create a date range
    date_range = pd.date_range(start=date_start, end=date_end, freq=temporal_resolution) # MS: month start frequency

    # Prepare the pattern for file names
    missing_files = []
    existing_files = []
    
    # Read the manifest file (assuming it's a list of filenames)
    with open(manifest_file, 'r') as f:
        manifest_files = set(f.read().splitlines())  # Assuming one filename per line

    # Loop through the date range and generate the file pattern for each month
    for date in date_range:
        # Format the date in YYYYMM
        YYYYMM = date.strftime('%Y%m')
        print(YYYYMM)
        YYYY = date.strftime('%Y')
        file_pattern = (
            f"http://wdc.dlr.de/C3S_312b_Lot2/AER/{parameter_code}/"
            f"{satellite}/{instrument}/TIR/{algorithm}/{version}/L3_MONTHLY/"
            f"{YYYY}/"
            f"{YYYYMM}-"
            f"C3S-L3_AEROSOL-AER_PRODUCTS-"
            f"{instrument}-{satellite}-{algorithm}-MONTHLY-{version}.nc"
        )

        # Check if the file is in the manifest
        if file_pattern not in manifest_files:
            missing_files.append(file_pattern)
        else:
            existing_files.append(file_pattern)

    if missing_files:
        nmissing = len(missing_files)
        print(f"Missing files for the following {nmissing} date(s):")
        for file in missing_files:
            print(file)
    else:
        print("All expected dates are present in the manifest file.")
        # print('Dates:', existing_files)
        
    return existing_files

file_list = check_files_in_manifest(delivery_start, delivery_end,temporal_resolution,satellite, instrument, algorithm, parameter_code,version, local_manif_fname)


# http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPA/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2011/201108-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPA-IMARS-MONTHLY-v7.1ALL.nc
# http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPA/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202502-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPAIMARS-MONTHLY-v7.1ALL.nc

202502
202503
202504
202505
202506
202507
202508
202509
202510
202511
202512
All expected dates are present in the manifest file.
Dates: ['http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202502-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202503-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202504-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202505-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202506-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IAS

In [19]:
print(file_list)

['http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202502-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202503-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202504-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202505-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202506-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b_Lot2/AER/DAOD/METOPC/IASI/TIR/IMARS/v7.1ALL/L3_MONTHLY/2025/202507-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc', 'http://wdc.dlr.de/C3S_312b

In [21]:
if download_delivery:
    for file in file_list:
        local_fname = work_dir + os.path.basename(file)
        print(local_fname)
        if os.path.exists(local_fname):
            print(f"Already exists: {local_fname}")
            continue
        print(f"Downloading {file}")
        with requests.get(file, stream=True) as r:
            r.raise_for_status()
            with open(local_fname, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        
if download_full_dataset:
    file_list_full = check_files_in_manifest(collection_start, collection_end, temporal_resolution,sensor_line,satellite, instrument, algorithm, version, local_manif_fname)
    for file in file_list_full:
        local_fname = work_dir + os.path.basename(file)
        print(local_fname)
        if os.path.exists(local_fname):
            print(f"Already exists: {local_fname}")
            continue
        print(f"Downloading {file}")
        with requests.get(file, stream=True) as r:
            r.raise_for_status()
            with open(local_fname, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                

/Users/cxjo/Documents/Datasets/AER/IMARS/202502-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
Already exists: /Users/cxjo/Documents/Datasets/AER/IMARS/202502-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
/Users/cxjo/Documents/Datasets/AER/IMARS/202503-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
Already exists: /Users/cxjo/Documents/Datasets/AER/IMARS/202503-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
/Users/cxjo/Documents/Datasets/AER/IMARS/202504-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
Already exists: /Users/cxjo/Documents/Datasets/AER/IMARS/202504-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
/Users/cxjo/Documents/Datasets/AER/IMARS/202505-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
Already exists: /Users/cxjo/Documents/Datasets/AER/IMARS/202505-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
/Users/cxjo/Documents/Datasets/A

In [22]:
fname = work_dir + os.path.basename(file_list[-1]) # last file of the list of existing files, for example.
print(fname)
nc = xr.open_dataset(fname)
print(nc.info())


/Users/cxjo/Documents/Datasets/AER/IMARS/202512-C3S-L3_AEROSOL-AER_PRODUCTS-IASI-METOPC-IMARS-MONTHLY-v7.1ALL.nc
xarray.Dataset {
dimensions:
	longitude = 360 ;
	latitude = 180 ;

variables:
	float32 total_number_of_observations(longitude, latitude) ;
		total_number_of_observations:units = 1 ;
		total_number_of_observations:long_name = Total Number of Observations ;
		total_number_of_observations:valid_range = [   0. 1000.] ;
	float32 retrieval entropy(longitude, latitude) ;
		retrieval entropy:units = 1 ;
		retrieval entropy:long_name = Retrieval Information Content ;
		retrieval entropy:valid_range = [0. 2.] ;
	float32 water_fraction(longitude, latitude) ;
		water_fraction:units = 1 ;
		water_fraction:long_name = Water Fraction ;
		water_fraction:valid_range = [0. 1.] ;
	float32 cloud_fraction(longitude, latitude) ;
		cloud_fraction:units = 1 ;
		cloud_fraction:long_name = Cloud Fraction ;
		cloud_fraction:valid_range = [0. 1.] ;
	float32 D_quality_flag(longitude, latitude) ;
		D_qua

In [36]:
# list of local filenames
flist_local = [work_dir + os.path.basename(file_list[i]) for i in range(len(file_list))]

def preprocess(ds):
    filename = ds.encoding["source"]
    yyyymm = os.path.basename(filename)[:6]
    time = pd.to_datetime(yyyymm, format="%Y%m")
    return ds.expand_dims(time=[time])

ds_all = xr.open_mfdataset(
    flist_local,
    preprocess=preprocess,
    concat_dim="time",
    combine="nested"
)

date_range = pd.date_range(start=delivery_start, end=delivery_end, freq=temporal_resolution) # MS: month start frequency
ds_all = ds_all.reindex(time=date_range)

# Extract arrays
aer = ds_all[parameter_list[0]]
print(ds_all[parameter_list[0]])
# raise SystemExit

lat = ds_all["latitude"].values
lon = ds_all["longitude"].values
times = ds_all["time"].values

# ----------------------------
# Create map figure
# ----------------------------
fig, ax = plt.subplots(
    figsize=(12, 5),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

# Initial frame
img = aer.isel(time=0).plot(
    ax=ax,
    x="longitude",
    y="latitude",
    cmap="viridis",
    transform=ccrs.PlateCarree(),
    add_colorbar=True
)

# Add coastlines
ax.coastlines()
# ax.add_feature(cfeature.BORDERS, linewidth=0.5)

def update(frame):
    ax.clear()

    aer.isel(time=frame).plot(
        ax=ax,
        x="longitude",
        y="latitude",
        cmap="viridis",
        transform=ccrs.PlateCarree(),
        add_colorbar=False
    )

    ax.coastlines()
    gl = ax.gridlines(draw_labels=True, linestyle="--")
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

    ax.set_title(f"{parameter_code} {algorithm}— {str(times[frame])[:10]}")

ani = FuncAnimation(
    fig,
    update,
    frames=len(times),
    interval=500,
    blit=False
)

plt.close(fig)

HTML(ani.to_jshtml())


<xarray.DataArray 'D_AOD550' (time: 11, longitude: 360, latitude: 180)> Size: 3MB
dask.array<concatenate, shape=(11, 360, 180), dtype=float32, chunksize=(1, 360, 180), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[us] 88B 2025-02-01 2025-03-01 ... 2025-12-01
  * longitude  (longitude) float32 1kB -179.5 -178.5 -177.5 ... 178.5 179.5
  * latitude   (latitude) float32 720B -89.5 -88.5 -87.5 ... 87.5 88.5 89.5
Attributes:
    units:        1
    long_name:    Dust Aerosol Optical Depth at 550nm
    valid_range:  [ 0. 10.]
